# ADS1002 Group Project

## Preprocessing 

In [2]:
#importing libraries 
import glob 
import pandas as pd 
import os

In [3]:
csv_demand = glob.glob("/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/ALL_DEMAND_DATA/*.csv")
df_demand = [pd.read_csv(file) for file in csv_demand]

combined_demand = pd.concat(df_demand, ignore_index = True) 

In [4]:
combined_demand.tail()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
1658960,QLD1,2008/04/30 22:00:00,6017.13,37.17,TRADE
1658961,QLD1,2008/04/30 22:30:00,5847.77,54.53,TRADE
1658962,QLD1,2008/04/30 23:00:00,5665.46,38.39,TRADE
1658963,QLD1,2008/04/30 23:30:00,5570.00,36.48,TRADE
1658964,QLD1,2008/05/01 00:00:00,5382.89,27.06,TRADE


### Reading in and combining all .txt files 

In [52]:
# Update to point at your local Temperature Data folder
TEMP_DIR = "/Users/thish/Documents/ADS1002/Datasets for temperature and energy demand modelling-20260731/Temperature Data"

# From the BoM Notes file -- 34 fields, confirmed via inspection that
# each station file has exactly 1 description/header line before data starts
TEMP_COLUMNS = [
    "record_id", "station_number",
    "year_local", "month_local", "day_local", "hour_local", "minute_local",
    "year_std", "month_std", "day_std", "hour_std", "minute_std",
    "precipitation_9am_mm", "precipitation_quality",
    "air_temperature_c", "air_temperature_quality",
    "wet_bulb_temp_c", "wet_bulb_quality",
    "dew_point_temp_c", "dew_point_quality",
    "relative_humidity_pct", "relative_humidity_quality",
    "wind_speed_kmh", "wind_speed_quality",
    "wind_direction_deg", "wind_direction_quality",
    "max_gust_kmh", "max_gust_quality",
    "mslp_hpa", "mslp_quality",
    "station_level_pressure_hpa", "station_level_pressure_quality",
    "aws_flag", "end_marker",
]

In [53]:
# Only grab real data files -- filters out __MACOSX/.DS_Store/._ junk that shows up if the folder was zipped/unzipped on a Mac
txt_temperature = glob.glob(os.path.join(TEMP_DIR, "*.txt"))
txt_temperature = [f for f in txt_temperature if not os.path.basename(f).startswith("._")]

print(f"Found {len(txt_temperature)} temperature station files:")
for f in txt_temperature:
    print(" -", os.path.basename(f))

Found 6 temperature station files:
 - HM01X_Data_094029_999999999743964.txt
 - HM01X_Data_086338_999999999743964.txt
 - HM01X_Data_023090_999999999743964.txt
 - HM01X_Data_066062_999999999743964.txt
 - HM01X_Data_086071_999999999743964.txt
 - HM01X_Data_040913_999999999743964.txt


In [54]:
df_temp_list = []
for file in txt_temperature:
    df = pd.read_csv(
        file,
        skiprows=1,             # skip the 1-line description/header row
        header=None,
        names=TEMP_COLUMNS,
        na_values=["", " "],    # blank fields (e.g. missing precipitation/AWS flag) -> NaN
        skipinitialspace=True,  # BoM pads values with leading spaces
        dtype=str,              # read as string first, convert explicitly below
                                 # (avoids the mixed-dtype warning from earlier)
    )
    df_temp_list.append(df)

combined_temp = pd.concat(df_temp_list, ignore_index=True)

# Drop columns with no analytical value
combined_temp = combined_temp.drop(columns=["record_id", "end_marker"])


In [57]:
numeric_cols = [
    "station_number",
    "year_local", "month_local", "day_local", "hour_local", "minute_local",
    "year_std", "month_std", "day_std", "hour_std", "minute_std",
    "precipitation_9am_mm", "air_temperature_c", "wet_bulb_temp_c",
    "dew_point_temp_c", "relative_humidity_pct", "wind_speed_kmh",
    "wind_direction_deg", "max_gust_kmh", "mslp_hpa",
    "station_level_pressure_hpa", "aws_flag",
]
for col in numeric_cols:
    combined_temp[col] = pd.to_numeric(combined_temp[col], errors="coerce")

# Quality flag columns stay as strings/categories (Y/N/W/S/I)
quality_cols = [c for c in combined_temp.columns if c.endswith("_quality")]
for col in quality_cols:
    combined_temp[col] = combined_temp[col].astype(str).str.strip()

In [58]:
combined_temp.head()

,station_number,year_local,month_local,day_local,hour_local,minute_local,year_std,month_std,day_std,hour_std,...,wind_speed_quality,wind_direction_deg,wind_direction_quality,max_gust_kmh,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag
0,94029,2000,1,1,2,0,2000,1,1,1,...,N,220.0,N,13.0,N,1019.3,N,1013.0,N,NaN
1,94029,2000,1,1,2,30,2000,1,1,1,...,N,240.0,N,11.2,N,1019.1,N,1012.8,N,NaN
2,94029,2000,1,1,3,0,2000,1,1,2,...,N,240.0,N,18.4,N,1018.9,N,1012.6,N,NaN
3,94029,2000,1,1,3,30,2000,1,1,2,...,N,240.0,N,18.4,N,1018.7,N,1012.4,N,NaN
4,94029,2000,1,1,4,0,2000,1,1,3,...,N,260.0,N,13.0,N,1018.5,N,1012.2,N,NaN


In [59]:
# Using LOCAL STANDARD TIME -- avoids the daylight-saving discontinuities
# the BoM Notes file warns about
combined_temp["datetime_std"] = pd.to_datetime(
    combined_temp[["year_std", "month_std", "day_std", "hour_std", "minute_std"]]
    .rename(columns={
        "year_std": "year", "month_std": "month", "day_std": "day",
        "hour_std": "hour", "minute_std": "minute",
    }),
    errors="coerce",
)

In [60]:
combined_temp.head()

,station_number,year_local,month_local,day_local,hour_local,minute_local,year_std,month_std,day_std,hour_std,...,wind_direction_deg,wind_direction_quality,max_gust_kmh,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std
0,94029,2000,1,1,2,0,2000,1,1,1,...,220.0,N,13.0,N,1019.3,N,1013.0,N,NaN,2000-01-01 01:00:00
1,94029,2000,1,1,2,30,2000,1,1,1,...,240.0,N,11.2,N,1019.1,N,1012.8,N,NaN,2000-01-01 01:30:00
2,94029,2000,1,1,3,0,2000,1,1,2,...,240.0,N,18.4,N,1018.9,N,1012.6,N,NaN,2000-01-01 02:00:00
3,94029,2000,1,1,3,30,2000,1,1,2,...,240.0,N,18.4,N,1018.7,N,1012.4,N,NaN,2000-01-01 02:30:00
4,94029,2000,1,1,4,0,2000,1,1,3,...,260.0,N,13.0,N,1018.5,N,1012.2,N,NaN,2000-01-01 03:00:00


In [61]:
quality_cols = [c for c in combined_temp.columns if c.endswith("_quality")]
for col in quality_cols:
    combined_temp[col] = combined_temp[col].astype(str).str.strip()
    combined_temp[col] = combined_temp[col].replace("nan", pd.NA)  # restore true missing values

In [62]:
print("Combined shape:", combined_temp.shape)
print("\nRows per station:")
print(combined_temp["station_number"].value_counts())
print("\nAny unparseable datetimes?", combined_temp["datetime_std"].isna().sum())
print("\nQuality flag value counts (air temperature):")
print(combined_temp["air_temperature_quality"].value_counts())

Combined shape: (1826244, 33)

Rows per station:
station_number
94029    386362
23090    355906
40913    350977
66062    350945
86071    262611
86338    119443
Name: count, dtype: int64

Any unparseable datetimes? 0

Quality flag value counts (air temperature):
air_temperature_quality
N    1825029
Name: count, dtype: int64


In [63]:
combined_temp.tail()

,station_number,year_local,month_local,day_local,hour_local,minute_local,year_std,month_std,day_std,hour_std,...,wind_direction_deg,wind_direction_quality,max_gust_kmh,max_gust_quality,mslp_hpa,mslp_quality,station_level_pressure_hpa,station_level_pressure_quality,aws_flag,datetime_std
1826239,40913,2020,1,20,7,0,2020,1,20,7,...,350.0,N,7.6,N,1007.9,N,1007.0,N,1.0,2020-01-20 07:00:00
1826240,40913,2020,1,20,7,30,2020,1,20,7,...,350.0,N,9.4,N,1008.0,N,1007.1,N,1.0,2020-01-20 07:30:00
1826241,40913,2020,1,20,8,0,2020,1,20,8,...,340.0,N,11.2,N,1008.2,N,1007.3,N,1.0,2020-01-20 08:00:00
1826242,40913,2020,1,20,8,30,2020,1,20,8,...,350.0,N,16.6,N,1008.2,N,1007.3,N,1.0,2020-01-20 08:30:00
1826243,40913,2020,1,20,9,0,2020,1,20,9,...,360.0,N,13.0,N,1008.1,N,1007.2,N,1.0,2020-01-20 09:00:00


In [38]:
combined_temp.isna().sum()

station_number                         0
year_local                             0
month_local                            0
day_local                              0
hour_local                             0
minute_local                           0
year_std                               0
month_std                              0
day_std                                0
hour_std                               0
minute_std                             0
precipitation_9am_mm               63715
precipitation_quality              63715
air_temperature_c                   1215
air_temperature_quality             1215
wet_bulb_temp_c                     4296
wet_bulb_quality                    4296
dew_point_temp_c                    1503
dew_point_quality                   1503
relative_humidity_pct               1506
relative_humidity_quality           1505
wind_speed_kmh                    444224
wind_speed_quality                444224
wind_direction_deg                445430
wind_direction_q

In [39]:
combined_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1826244 entries, 0 to 1826243
Data columns (total 34 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   station_number                  int64         
 1   year_local                      int64         
 2   month_local                     int64         
 3   day_local                       int64         
 4   hour_local                      int64         
 5   minute_local                    int64         
 6   year_std                        int64         
 7   month_std                       int64         
 8   day_std                         int64         
 9   hour_std                        int64         
 10  minute_std                      int64         
 11  precipitation_9am_mm            float64       
 12  precipitation_quality           object        
 13  air_temperature_c               float64       
 14  air_temperature_quality         object        
 15

In [40]:
#combined_temp.to_csv("combined_temp.csv.gz", index=False, compression="gzip")